# 02 · Logistic regression: normalisation, regularisation, feature selection

The industry baseline (benchmarking doc). Questions:
1. **Does normalisation matter, and which?** (LR-01 … LR-05, no penalty, all features)
2. **Does regularisation help: L2, L1, elastic net?** (LR-06 … LR-08, strength tuned by nested CV)
3. **Is training only on the important features better?** (LR-09 … LR-11, LR-15, LR-16)
4. **Which feature set transfers, and do curves or class weights help?** (LR-12 … LR-14)

All runs: the same 25 grouped folds; scaling and selection refitted inside each training fold.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "02"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

## 1. Normalisation (no penalty, FS_ALL)

In [ ]:
for rid, kind in [("LR-01", "none"), ("LR-02", "z"), ("LR-03", "log_z"), ("LR-04", "log_robust"), ("LR-05", "quantile")]:
    run(rid, M.pipeline(M.preprocess(kind), M.logreg()), "FS_ALL", f"No penalty · {M.PREPROCESSORS[kind]}")
b = M.leaderboard(notebook=NB).query("run_id in ['LR-01', 'LR-02', 'LR-03', 'LR-04', 'LR-05']")
BEST_PRE = {"LR-01": "none", "LR-02": "z", "LR-03": "log_z", "LR-04": "log_robust", "LR-05": "quantile"}[b.run_id.iloc[0]]
print("best preprocessing:", BEST_PRE)

## 2. Regularisation (penalty strength tuned inside each training fold)

In [ ]:
P = lambda: M.preprocess(BEST_PRE)
C_GRID = {"logisticregression__C": loguniform(1e-3, 1e2)}
run("LR-06", M.tuned(M.pipeline(P(), M.logreg(C=1.0)), C_GRID), "FS_ALL", f"L2 (ridge), C tuned · {BEST_PRE}")
run("LR-07", M.tuned(M.pipeline(P(), M.logreg(C=1.0, l1_ratio=1)), C_GRID), "FS_ALL", f"L1 (lasso), C tuned · {BEST_PRE}")
run("LR-08", M.tuned(M.pipeline(P(), M.logreg(C=1.0, l1_ratio=.5)), {**C_GRID, "logisticregression__l1_ratio": uniform(.05, .9)}),
    "FS_ALL", f"Elastic net, C and L1 ratio tuned · {BEST_PRE}");

### The L1 path: how many features does the model need?
Stronger L1 (smaller C) drops more features to exactly zero. Each point = mean AUC over the 25 folds.

In [ ]:
path = []
for C in np.logspace(-3, 1, 14):
    f = M.run_experiment("path", M.pipeline(P(), M.logreg(C=C, l1_ratio=1)), dev, "FS_ALL", notebook="tmp",
                         description="", splits=SPLITS, log=False, verbose=False)
    nz = f.features_used.str.split(";").map(len)
    path.append({"C": C, "auc": f.auc.mean(), "features kept (L1 non-zero)": (f.n_features_used).mean()})
path = pd.DataFrame(path)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].semilogx(path.C, path.auc, "o-"); ax[0].set(xlabel="C (larger = weaker penalty)", ylabel="AUC", title="AUC along the L1 path")
ax[1].plot(path["features kept (L1 non-zero)"], path.auc, "o-"); ax[1].set(xlabel="features with a non-zero weight", ylabel="AUC", title="AUC vs number of features")
plt.tight_layout(); path.round(3)

## 3. Train only on the important features

In [ ]:
run("LR-09", M.tuned(M.pipeline(P(), M.select_l1(), M.logreg()), {"selectfrommodel__estimator__C": loguniform(1e-3, 1e1)}),
    "FS_ALL", "L1 picks features (C tuned), then refit WITHOUT penalty on those only")
run("LR-10", M.tuned(M.pipeline(P(), M.logreg(C=1.0)), C_GRID), "FS_PRUNED", "L2 on FS_PRUNED (duplicates removed by hand)")
run("LR-10b", M.tuned(M.pipeline(P(), M.CorrelationPruner(.9), M.logreg(C=1.0)), C_GRID), "FS_ALL", "L2 after per-fold correlation pruning (|rho|>0.9)")
run("LR-11", M.pipeline(P(), M.StabilitySelector(n_boot=50), M.logreg(C=1.0)), "FS_ALL", "Stability selection (L1 on 50 half-samples, kept if chosen ≥70%), then L2")
run("LR-15", M.tuned(M.pipeline(P(), M.select_mi(), M.logreg(C=1.0)), {**C_GRID, "selectkbest__k": randint(3, 15)}), "FS_ALL", "Mutual-information top-k (k tuned), then L2")
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
rfe = RFECV(M.logreg(C=1.0), step=1, cv=StratifiedKFold(3, shuffle=True, random_state=M.SEED), scoring="roc_auc", min_features_to_select=2)
run("LR-16", M.pipeline(P(), rfe, M.logreg(C=1.0)), "FS_ALL", "Recursive feature elimination with CV (RFECV), then L2");

### How stable is the selection?
Share of the 25 folds in which each feature was kept. A feature kept in only a few folds isn't reliably useful.

In [ ]:
r = M.read_results(); freq = {}
for rid in ["LR-07", "LR-09", "LR-11", "LR-15", "LR-16"]:
    used = r[r.run_id == rid].features_used.str.split(";")
    freq[rid] = pd.Series([f for u in used for f in u]).value_counts() / len(used)
freq = pd.DataFrame(freq).fillna(0).sort_values("LR-11", ascending=False)
plt.figure(figsize=(8, 0.3 * len(freq) + 1)); sns.heatmap(freq, annot=True, fmt=".0%", cmap="Blues", cbar=False)
plt.title("Share of folds in which a feature was selected"); plt.show()

## 4. Transferable features, curves and class weights

In [ ]:
run("LR-12", M.tuned(M.pipeline(P(), M.logreg(C=1.0)), C_GRID), "FS_TRANSFER", "L2 on FS_TRANSFER (features that exist in the real data)")
from sklearn.preprocessing import SplineTransformer
run("LR-13", M.tuned(M.pipeline(P(), SplineTransformer(n_knots=4, degree=3), M.logreg(C=1.0)), C_GRID), "FS_TRANSFER",
    "L2 with a spline curve per feature (GAM-like), FS_TRANSFER")
run("LR-14", M.tuned(M.pipeline(P(), M.logreg(C=1.0, class_weight="balanced")), C_GRID), "FS_TRANSFER", "L2 with balanced class weights, FS_TRANSFER");

## 5. Leaderboard and significance

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
r = M.read_results(); r = r[r.notebook == NB]
plt.figure(figsize=(10, 0.35 * r.run_id.nunique() + 1.2))
sns.boxplot(data=r, y="run_id", x="auc", order=board.run_id, color="#8DA0CB")
plt.axvline(M.leaderboard().query("run_id == 'REF-LR'").auc.iloc[0], ls="--", c="#C44E52", label="REF-LR (notebook 01)")
plt.legend(); plt.title("AUC across the same 25 folds"); plt.ylabel(""); plt.show()

In [ ]:
best = board.run_id.iloc[0]
pd.DataFrame([M.corrected_ttest(best, rid) for rid in board.run_id[1:]]).round(4)

### What the transferable model says (LR-12 refitted on all training rows; standardised coefficients)

In [ ]:
m = M.tuned(M.pipeline(P(), M.logreg(C=1.0)), C_GRID).fit(*M.xy(dev, M.FS_TRANSFER))
lr = m.best_estimator_[-1]; names = m.best_estimator_[:-1].get_feature_names_out()
coef = pd.Series(lr.coef_[0], index=names).sort_values()
plt.figure(figsize=(7, 0.3 * len(coef) + 1)); coef.plot.barh(color=np.where(coef > 0, "#55A868", "#C44E52"))
plt.title(f"Coefficients (C = {m.best_params_['logisticregression__C']:.3g}); positive = more likely to pay"); plt.show()

## Findings (Salud-tuned world)
- **Scaling:** log1p plus z or robust scaling beat both raw and z-only. LR-03 and LR-04 reach 0.873, against 0.861 for raw (LR-01) and z-only (LR-02). Quantile scaling (LR-05) reaches 0.864.
- **Penalties tie:** L2 (LR-06) 0.875, elastic net 0.875, L1 0.875. Corrected t-tests give p > 0.39.
  L1 keeps about 17 of 24 features. Along the L1 path, 4 features already give 0.868 and 9 give 0.871.
- **Fewer features:** refitting only on the L1-chosen features (LR-09, 13.6 features) gives 0.874, the same AUC with fewer inputs.
  Stability selection keeps only about 3 features and still reaches 0.866.
- **FS_TRANSFER** (LR-12, 0.866) costs about 0.009 AUC compared with all features (p = 0.01). That is the price of using only features that exist in Salud's data.
- **Balanced class weights** (LR-14) leave the AUC unchanged and make the Brier score worse (0.138). Don't use them.